# VRDFormer Training on Google Colab

This notebook trains the full VRDFormer pipeline on VidOR using a Colab T4 GPU.

**Prerequisites:**
- VRDFormer repo in Google Drive at `/content/drive/MyDrive/VRDFormer/`
- Data prepared (Phase 1) — annotation pickle and frame indices in `data/metadata/`
- DETR weights downloaded (Phase 2) — `data/weights/detr-r101-2c7b67e5.pth`
- Config files created (Phase 2) — `configs/vidor_colab_stage1.json` and `configs/vidor_colab_stage2.json`

## 1. Mount Google Drive & Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Install dependencies (torch/torchvision pre-installed on Colab with CUDA)
!pip install decord timm scipy lap pycocotools -q

In [ ]:
import os
import sys
import torch

# Navigate to the VRDFormer repo
%cd /content/drive/MyDrive/VRDFormer

# Verify GPU
!nvidia-smi
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
print(f'GPU: {torch.cuda.get_device_name(0)}')

## 2. Verify Data

Check that all prepared data files exist before starting training.

In [ ]:
# Check key files exist
import os

required = [
    'data/metadata/vidor_annotations.pkl',
    'data/metadata/vidor_train_frames_stage1.json',
    'data/metadata/vidor_train_frames_stage2.json',
    'data/metadata/vidor_val_frames.json',
    'data/weights/detr-r101-2c7b67e5.pth',
    'configs/vidor_colab_stage1.json',
    'configs/vidor_colab_stage2.json',
]

all_ok = True
for path in required:
    exists = os.path.exists(path)
    size = os.path.getsize(path) / 1e6 if exists else 0
    status = f'✅ {size:.1f} MB' if exists else '❌ MISSING'
    print(f'  {path:<55} {status}')
    if not exists:
        all_ok = False

if all_ok:
    print('\nAll files present — ready to train!')
else:
    print('\n⚠️  Missing files — check your data preparation.')

## 3. Stage 1: Smoke Test (Optional)

Run 2 epochs on 100 videos to verify the pipeline works. Skip if you've already tested locally.

In [ ]:
# Smoke test — uncomment to run
# !python -m torch.distributed.launch \
#     --master_port 47749 \
#     --nproc_per_node=1 \
#     main.py \
#     --accumulate_steps 1 \
#     --lr_backbone 1e-5 \
#     --lr 5e-5 \
#     --num_queries 200 \
#     --dataset_config configs/vidorpart_local_stage1.json \
#     --epochs 2
print('Smoke test skipped — uncomment cell to run')

## 4. Stage 1: Full Training

Train the pair detection + tracking model. This is the main training phase.

In [ ]:
%%time
# Stage 1 full training (5 epochs, ~4-6 hours on T4)
!python -m torch.distributed.launch \
    --master_port 47749 \
    --nproc_per_node=1 \
    main.py \
    --accumulate_steps 1 \
    --lr_backbone 1e-5 \
    --lr 5e-5 \
    --num_queries 200 \
    --dataset_config configs/vidor_colab_stage1.json

### Verify Stage 1 Output

In [ ]:
import os, glob
ckpts = sorted(glob.glob('data/ckpts/vidor_stage1/checkpoint*.pth'))
print(f'Stage 1 checkpoints: {len(ckpts)}')
for ckpt in ckpts:
    size_mb = os.path.getsize(ckpt) / 1e6
    print(f'  {ckpt:<50} {size_mb:.1f} MB')

## 5. Stage 2: Relation Classification

Train the temporal relation classifier using Stage 1 output.

In [ ]:
%%time
# Stage 2 training (2 epochs, ~1-2 hours on T4)
!python -m torch.distributed.launch \
    --master_port 47745 \
    --nproc_per_node=1 \
    main.py \
    --accumulate_steps 1 \
    --lr_backbone 1e-5 \
    --lr 5e-5 \
    --num_queries 200 \
    --dataset_config configs/vidor_colab_stage2.json

## 6. Evaluation

Evaluate the trained Stage 2 model on the validation set.

In [ ]:
!python -m torch.distributed.launch \
    --nproc_per_node=1 \
    main.py \
    --eval \
    --dataset_config configs/vidor_colab_stage2.json \
    --resume data/ckpts/vidor_stage2/checkpoint.pth

## 7. Summary

After successful completion:
- **Stage 1 checkpoint:** `data/ckpts/vidor_stage1/checkpoint0004.pth`
- **Stage 2 checkpoint:** `data/ckpts/vidor_stage2/checkpoint.pth`
- These checkpoints can be downloaded and used with `inference.py` for inference on new videos.

**Next:** Copy checkpoints to local machine and run Phase 5 (inference).